# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: LoRA
* Model: GPT-2
* Evaluation approach: accuracy
* Fine-tuning dataset: imbd

In [12]:
!pip install transformers datasets evaluate accelerate peft
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import evaluate

In [13]:
pip install bitsandbytes

Note: you may need to restart the kernel to use updated packages.


## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [29]:
# 1. Install and Import
try:
    import evaluate
except ImportError:
    !pip install -q transformers datasets evaluate accelerate

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import evaluate

# Define constants
MODEL_NAME = "gpt2"
DATASET_NAME = "imdb"

In [30]:
# 2. Load Dataset and Tokenizer
dataset = load_dataset(DATASET_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token # GPT-2 specific requirement

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

# Create evaluation subset (500 samples)
test_subset = dataset["test"].shuffle(seed=42).select(range(500))
tokenized_test = test_subset.map(preprocess_function, batched=True)

In [31]:
# 3. Load Foundation Model for Sequence Classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

# Align model padding with tokenizer
model.config.pad_token_id = model.config.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
# 4. Set up Evaluation Metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
# 5. Perform Initial Evaluation
trainer = Trainer(
    model=model,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Evaluating baseline foundation model...")
baseline_results = trainer.evaluate()
print(f"Foundation Model Accuracy: {baseline_results['eval_accuracy']:.2%}")

## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights

In [33]:
from peft import LoraConfig, get_peft_model, TaskType

# 1. Define LoRA Config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1, 
    target_modules=["c_attn"],
    fan_in_fan_out=True # Critical for GPT-2 Conv1D layers
)

# 2. Wrap the model
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377


In [34]:
# Create a small training subset (1000 samples)
train_subset = dataset["train"].shuffle(seed=42).select(range(1000))

def preprocess_for_peft(examples):
    result = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
    result["labels"] = examples["label"]
    return result

tokenized_train = train_subset.map(preprocess_for_peft, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [35]:
training_args = TrainingArguments(
    output_dir="./gpt2-lora-results",
    learning_rate=2e-4, 
    per_device_train_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test, 
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.665825,0.608000


TrainOutput(global_step=250, training_loss=0.802950927734375, metrics={'train_runtime': 5280.5208, 'train_samples_per_second': 0.189, 'train_steps_per_second': 0.047, 'total_flos': 262207438848000.0, 'train_loss': 0.802950927734375, 'epoch': 1.0})

In [36]:
# Save the fine-tuned adapter weights
peft_model.save_pretrained("gpt2-lora-adapter")

print("PEFT Model saved successfully!")

PEFT Model saved successfully!


###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [38]:
# Saving the model
peft_model.save_pretrained("/tmp/peft-model")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.